In [ ]:
import pandas as pd
import numpy as np
import missingno as msn
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('reviews_to_2020.csv')
print(len(df))
df.head()

In [ ]:
print(df.info())
print(df.describe())
print(df.columns)

In [ ]:
# rename columns 
df.columns = df.columns.str.lower().str.replace(".", "", regex=False)
df.columns

In [ ]:
# let's first clean up the agtron
df[["agtron_1", "agtron_2"]] = df["agtron"].str.split("/", expand=True)
df = df.drop(["agtron"], axis=1)
df.head() 

In [ ]:
# standard prices to per oz
import re

def standardize_pricing(price_string):
    """
    Standardizes a coffee price string to USD per ounce.

    Args:
        price_string: A string representing the coffee price,
                      e.g., '$xx.xx/y [ounces or grams]' or
                      'NT $xx.xx/y [ounces or grams]'.

    Returns:
        The price per ounce in USD as a float, or None if the
        string is not in the expected format.
    """
    if not isinstance(price_string, str):
        return None
    
    
    pattern = re.compile(r"""
        (?P<currency>NT\s\$|\$)? # optional currency (NT or USD)
        \s* # optional whitespace
        (?P<price>[\d,]+(?:\.\d+)?) # the price
        / # seperator
        (?P<weight>\d+) # weight
        \s* # optional whitespace
        (?P<unit>ounces|grams)                     
    """, re.VERBOSE | re.IGNORECASE)
    
    match = pattern.match(price_string)
    
    if not match:
        return None
    
    parts = match.groupdict()
    
    price = float(parts["price"].replace(",", ""))
    weight = float(parts["weight"])
    unit = parts["unit"].lower()
    currency = "USD" if parts["currency"] == "$" else "NT"
    
    if unit == 'grams':
        weight_oz = weight / 28.34952
    else:
        weight_oz = weight
        
    if weight_oz == 0:
        return None
    
    # 6/9/25
    ntd_to_usd_rate = 29.9340
    price_usd = price
    
    if currency == "NT":
        price_usd /= ntd_to_usd_rate
    
    price_per_oz = price_usd / weight_oz
    
    return price_per_oz


In [ ]:
df["price_per_oz"] = np.round(df["est price"].apply(standardize_pricing), 2)
display(df["price_per_oz"][:20])
display(df['est price'][:20])
df = df.drop(['est price'], axis=1)

In [ ]:
df['coffee origin'][:10]

In [ ]:
known_origins = [
    'north america', 
    'central america',
    'south america',
    'asia',
    'costa rica', 
    'el salvador', 
    'puerto rico',
    'united states',
    'papua new guinea',
    'dominican republic',
    'colombia',
    'ethiopia',
    'guatemala',
    'honduras',
    'indonesia',
    'nicaragua',
    'thailand',
    'tanzania',
    'vietnam',
    'mexico',
    'brazil',
    'panama',
    'hawaii', 
    'kenya',
    'rwanda',
    'china', 
    'india',
    'peru',
    'yemen',
    'burundi',
    'taiwan',
    'st. helena',
    'ecuador',
    'zambia',
    'haiti',
    'hawaii',
    'south africa',
    'uganda',
    'malaysia',
    'latin america',
    'laos',
    'congo',
    'venezuela',
    'bolivia',
    'philippines',
    'new guinea',
    'indo-pacific',
    'africa',
    'jamaica',
    'sumatra',
    'guji',
    'gedeo',
    'apaneca'
]

def extract_origins(origin_string, known):
    """
    Extracts a list of known countries or regions from a string.

    Args:
        origin_string (str): The string from the 'coffee origin' column.
        known_list (list): A list of countries to search for.

    Returns:
        list: A list of unique countries found in the string.
    """
    if not isinstance(origin_string, str):
        return ["unkown"]
    elif origin_string == "Not disclosed":
        return ["unkown"]
    
    found_origins = set()
    
    potential_origins = origin_string.replace("’", "").replace("ʻ", "").replace("'", "").replace("‘", "").split(";")
    
    for part in potential_origins:
        for origin in known:
            if re.search(r"\b" + re.escape(origin) + r"\b", part, re.IGNORECASE):
                found_origins.add(origin)
    
    if len(found_origins) == 0:
        return ["unkown"] 
    
    return sorted(list(found_origins))

In [ ]:
df["coffee origin"] = df["coffee origin"].str.lower()
df["countries_extracted"] = df["coffee origin"].apply(lambda origin: extract_origins(origin, known_origins))

display(df[["countries_extracted", "coffee origin"]][:20])

unkown_origins = df[(df["countries_extracted"].apply(lambda lst: lst == ["unkown"])) & (df["coffee origin"] != "not disclosed") & (df["coffee origin"].notna())]
print(len(unkown_origins))
display(unkown_origins)
# unkown_origins.to_csv("unkown_origins.csv")
# print(df.iloc[757]["coffee origin"])

# df = df.dropna(subset=["coffee origin"])

# display(df.iloc[[379, 633], :])
# print(df.iloc[[379, 633], :]["countries_extracted"])



In [ ]:
# Check if rows with null "acidity" have non-null "with milk"
mask = df["acidity"].isnull()
result = df.loc[mask, "with milk"].notnull().all()
print("All null acidity rows have non-null 'with milk':", result)
print("Count of null acidity rows:", mask.sum())
print("Count of null acidity rows with non-null 'with milk':", df.loc[mask, "with milk"].notnull().sum())
display(df[df["acidity"].notnull() & df["with milk"].notnull()][["blind assessment", "notes"]].head(10))
display(df[df["acidity"].isnull() & df["with milk"].notnull()][["blind assessment", "notes"]].head(10))
# print(len(df[df["acidity"].notnull()]))

In [ ]:
# Confirm "Evaluated as espresso" when acidity is null and with milk is not null
espresso_mask = df["acidity"].isnull() & df["with milk"].notnull()
espresso_regex = r"espresso"
espresso_present = df.loc[espresso_mask, "blind assessment"].str.contains(espresso_regex, case=False, regex=True).all()
print(f'"Evaluated as espresso" present for all (acidity null & with milk not null): {espresso_present}')

# Confirm "tested cold" when acidity is not null and with milk is not null
cold_mask = df["acidity"].notnull() & df["with milk"].notnull()
cold_regex = r"tested cold"
cold_present = df.loc[cold_mask, "blind assessment"].str.contains(cold_regex, case=False, regex=True).all()
print(f'"tested cold" present for all (acidity not null & with milk not null): {cold_present}')

acidity_wmilk_mask = df["blind assessment"].str.contains("evaluated as espresso", case=False, regex=True).fillna(False)

# Check 1: In the rows where the phrase is present, is 'acidity' always null?
is_acidity_null = df.loc[acidity_wmilk_mask, 'acidity'].isnull().all()

# Check 2: In those same rows, is 'with milk' always not null?
is_milk_not_null = df.loc[acidity_wmilk_mask, 'with milk'].notnull().all()

# Print the results
print(f"Checking the inverse: when 'blind assessment' contains 'evaluated as espresso'...")
print(f"- Is 'acidity' always null? {is_acidity_null}")
print(f"- Is 'with milk' always not null? {is_milk_not_null}")

# You can combine them for a final boolean confirmation
if is_acidity_null and is_milk_not_null:
    print("\nConfirmation: The relationship holds true.")
else:
    print("\nConfirmation: The relationship does not hold for all cases.")

display(df[df["blind assessment"].str.contains("evaluated as espresso", case=False, regex=True) & df["with milk"].isnull()].head(10))


In [ ]:
# Condition 1: Check if 'blind assessment' contains "evaluated as espresso" and "tested cold"
is_espresso = df["blind assessment"].str.contains("espresso", case=False, regex=True).fillna(False)
is_cold = df["blind assessment"].str.contains("cold", case=False, regex=True).fillna(False)

# Condition 2: Check if 'with milk' is not null
has_milk = df["with milk"].notnull()

# Create one-hot encoded columns for test types
# espresso cases
df["test_method"] = np.where(
    is_espresso & ~has_milk, 
    "espresso_black",
    np.where(
        is_espresso & has_milk, 
        "espresso_with_milk",
        np.where(
            is_cold & ~has_milk, 
            "cold_black",
            np.where(
                is_cold & has_milk, 
                "cold_with_milk",
                np.where(
                    ~is_espresso & ~is_cold & ~has_milk, 
                    "hot_black",
                    np.where(
                        ~is_espresso & ~is_cold & has_milk, 
                        "hot_with_milk", 
                        "unknown"
                    )
                )
            )
        )
    )
)
print(df["test_method"].value_counts())
display(df.head())
# df["espresso_black"] = (is_espresso & ~has_milk).astype(int)
# df["espresso_with_milk"] = (is_espresso & has_milk).astype(int)

# # cold test cases
# df["cold_black"] = (is_cold & ~has_milk).astype(int)
# df["cold_with_milk"] = (is_cold & has_milk).astype(int)

# # hot test cases 
# df["hot_black"] = (~is_espresso & ~is_cold & ~has_milk).astype(int)
# df["hot_with_milk"] = (~is_espresso & ~is_cold & has_milk).astype(int)
# display(df[df["hot_with_milk"] == True][["blind assessment", "review date"]].head())

In [ ]:
# extract processing method
PROCESS_KEYWORDS = {
    "Honey": ["honey", "pulped natural", "semi-washed"],
    "Anaerobic": ["anaerobic", "carbonic maceration", "carbonic", "experimental"],
    "Natural": ["natural", "dry", "sun dried"],
    "Washed": ["washed", "wet washed"],
}

def extract_process(notes):
    """
    Extracts the processing method from the notes based on predefined keywords.

    Args:
        notes (str): The notes from the coffee review.

    Returns:
        str: The processing method found in the notes, or "Unkown" if no method is found.
    """
    if not isinstance(notes, str):
        return ["unknown"]
    
    processes = set()
    for process, keywords in PROCESS_KEYWORDS.items():
        for keyword in keywords:
            if re.search(r"\b" + re.escape(keyword) + r"\b", notes, re.IGNORECASE):
            # if keyword in notes.lower():
                processes.add(process.lower())
    
    if len(processes) == 0:
        return ["unknown"]
    
    return list(processes)

df["process"] = df["notes"].apply(extract_process)

display(df[["process", "notes"]].head(10))
print(df["process"].value_counts())
df[df["process"] == "unknown"].to_csv("unknown_process.csv")

df[df["process"].apply(lambda lst: len(lst) > 1)].to_csv("multiple_process.csv")

# display(df[df["process"].apply(lambda lst: lst == ["Honey", "Anaerobic", "Washed"])].loc[745, "notes"])


In [ ]:
# roaster location is really only relevant if the user wants something close to them. It makes more sense to extract cities then
# COME BACK LATER
df[df["roaster location"].apply(lambda x: x == "Mexico; Ethiopia; Brazil")]["url"]

In [ ]:
# give a relative, natural language scale for pricing
df["review date"] = pd.to_datetime(df["review date"], errors="coerce")
df["year"] = df["review date"].dt.year.astype(int)

def calculate_yearly_tiers(group):
    """
    Calculates price tiers within a single year, into 4 quartiles
    """
    return pd.qcut(
        group["price_per_oz"],
        q=4,
        labels=["value", "standard", "premium", "luxury"],
        duplicates="drop"
    )

df["price_tier"] = df.groupby("year").apply(calculate_yearly_tiers).reset_index(level=0, drop=True)

df.head()
# df.to_csv("data_so_far.csv")

In [ ]:
# extract flavor notes
# define keyword dictionary
FLAVOR_KEYWORDS = {
    # Positive Flavors
    'Fruity': [
        'strawberry', 'raspberry', 'blueberry', 'blackberry', 'marionberry', 'cranberry', # Berry
        'raisin', 'prune', 'date', 'fig', # Dried Fruit
        'grapefruit', 'orange', 'lemon', 'lime', 'tangerine', 'bergamot', 'zest', # Citrus
        'cherry', 'black cherry', 'peach', 'apricot', 'plum', 'nectarine', # Stone Fruit
        'pineapple', 'mango', 'passion fruit', 'guava', 'lychee', 'kiwi', 'papaya', 'coconut', # Tropical
        'apple', 'red apple', 'green apple', 'pear', 'grape', 'concord grape', 'white grape', 'melon', 'pomegranate', # Other Fruit
        'fruit'
    ],
    'Floral': [
        'jasmine', 'rose', 'hibiscus', 'lavender', 'chamomile', 'honeysuckle', 'orange blossom', 'elderflower', # Aromatic Flowers
        'floral'
    ],
    'Sweet': [
        'molasses', 'maple syrup', # Syrupy
        'brown sugar', 'caramel', 'butterscotch', 'toffee', 'nougat', 'marshmallow', 'cane sugar', # Sugars
        'vanilla', 'cream', 'custard', 'marzipan', # Confectionary
        'honey', 'honeydew', # Honey
        'sweet'
    ],
    'Nutty/Cocoa': [
        'almond', 'hazelnut', 'peanut', 'walnut', 'pecan', 'cashew', 'praline', # Nutty
        'chocolate', 'milk chocolate', 'dark chocolate', 'baker\'s chocolate', 'cacao nibs', # Cocoa
        'nutty', 'cocoa'
    ],
    'Spicy': [
        'cinnamon', 'nutmeg', 'clove', 'cardamom', 'allspice', 'gingerbread', # Baking Spices
        'anise', 'licorice', 'black pepper', 'coriander', 'ginger', # Pungent Spices
        'spice'
    ],
    'Roasted/Toasted': [
        'grain', 'malt', 'oatmeal', 'toast', 'sourdough bread', 'brown rice', # Cereal
        'smoke', 'ash', 'acrid', 'burnt sugar', 'char', # Burnt
        'pipe tobacco', 'cigar', 'cedar', # Tobacco
        'roasted', 'toasted'
    ],
    'Earthy/Herbal': [
        'fresh-cut grass', 'hay', 'bell pepper', 'olive', 'tomato', 'pea', # Green/Vegetative
        'black tea', 'green tea', 'oolong tea', 'mint', 'thyme', 'lemongrass', 'rooibos', # Herbal/Tea-like
        'earthy', 'herbal', 'tea-like'
    ],
    'Winey/Fermented': [
        'red wine', 'white wine', 'champagne', 'boozy', 'whiskey', 'rum', # Winey/Alcoholic
        'fermented'
    ],
    'Savory': ['umami', 'soy sauce', 'leather', 'meaty'],

    # Other Attributes
    'Mouthfeel': [
        'light-bodied', 'medium-bodied', 'full-bodied', 'heavy', 'thin', 'watery', # Body
        'creamy', 'buttery', 'silky', 'smooth', 'juicy', 'syrupy', 'velvety', 'rich', 'delicate', 'astringent', 'gritty' # Texture
    ],
    'Acidity': [
        'mild', 'soft', 'mellow', 'delicate', # Intensity
        'bright', 'crisp', 'tart', 'tangy', 'vibrant', 'lively', 'sparkling', 'effervescent', 'complex', 'structured' # Quality
    ],
    'Aftertaste': ['clean', 'lingering', 'long', 'quick', 'short', 'dry', 'sweet', 'cloying'],
    
    # Negative Flavors/Defects
    'Defect/Negative': [
        'earthy', 'damp soil', 'mushroom', 'musty', 'moldy', # Earthy/Musty
        'rubbery', 'petroleum', 'plastic', 'bitter medicine', 'iodine', 'phenolic', # Chemical
        'cardboard', 'paper', 'stale', 'woody', 'sawdust', # Woody/Papery
        'over-fermented', 'sour', 'vinegary', 'alcoholic', # Fermented Taints
        'grassy', 'beany', 'under-ripe', # Green/Unripe
        'baggy', 'hidy', 'scorched', 'tipped' # Other
    ]
}

In [ ]:
def extract_structured_profile(text):
    """
    Extracts a dictionary of flavor categories mapped to the specific
    keywords found in the text.
    """
    if not isinstance(text, str):
        return {}
    
    found_profile = {}
    lower_text = text.lower()
    
    for category, keywords in FLAVOR_KEYWORDS.items():
        found_keywords = []
        for keyword in keywords:
            # Use regex with word boundaries (\b) to avoid matching parts of words
            if re.search(r'\b' + re.escape(keyword) + r'\b', lower_text):
                found_keywords.append(keyword)
        
        if found_keywords:
            found_profile[category] = sorted(list(set(found_keywords)))
                
    return found_profile

# Apply the function to create the new structured profile column
df['flavor_profile'] = df['blind assessment'].apply(extract_structured_profile)

# For display and potential use in prompts, create a formatted string version
def format_profile_for_llm(profile_dict):
    if not profile_dict:
        return ""
    parts = []
    for category, notes in profile_dict.items():
        notes_str = ", ".join(notes)
        parts.append(f"{category} ({notes_str})")
    return "; ".join(parts)

df['flavor_profile_str'] = df['flavor_profile'].apply(format_profile_for_llm)
df.to_csv("data_so_far.csv")


In [ ]:
# extract the varietal
# This set includes common varietals, hybrids, and alternative spellings.
COFFEE_VARIETALS = {
    # Common Arabica
    'typica', 'bourbon', 'caturra', 'catuai', 'geisha', 'gesha', 'pacamara', 
    'pacas', 'maragogipe', 'mundo novo', 'kent', 's795', 'jember', 'villa sarchi',
    'sl28', 'sl34', 'sl14', 'batian', 'ruiru 11', 'blue mountain', 'sumatra',
    'timor hybrid', 'hibrido de timor', 'catimor', 'castillo', 'colombia',
    'sarchimor', 'ihcafe 90', 'lempira', 'parainema', 'centroamericano',
    'mokka', 'mocha', 'java', 'kona', 'yellow bourbon', 'red bourbon', 
    'pink bourbon', 'orange bourbon', 'yellow caturra', 'red caturra',
    'yellow catuai', 'red catuai', 'maracaturra', 'wush wush', 'sidra', 'sudanese rumé',
    
    # Ethiopian Heirlooms (often referred to as a group)
    'heirloom', 'ethiopian heirloom', 'kurume', 'daga', 'wolisho',

    # Common Robusta
    'robusta', 'congensis', 'canephora',

    # Other Species
    'liberica', 'excelsa', 'stenophylla', 'arabica' # Include base species
}

def extract_varietals(text):
    """
    Extracts a list of coffee varietals from a text string.
    """
    if not isinstance(text, str):
        return ["unkown"]
    
    found_varietals = set()
    lower_text = text.lower()
    
    for varietal in COFFEE_VARIETALS:
        # Use regex with word boundaries (\b) to avoid matching parts of words.
        # This handles cases like 'java' not matching in 'javascript'.
        # re.escape handles special characters in the varietal name.
        if re.search(r'\b' + re.escape(varietal) + r'\b', lower_text, re.IGNORECASE):
            # Standardize spellings (e.g., Gesha/Geisha)
            if varietal == 'gesha':
                found_varietals.add('geisha')
            elif varietal == 'hibrido de timor':
                found_varietals.add('timor hybrid')
            else:
                found_varietals.add(varietal)
    if len(found_varietals) == 0:
        return ["unkown"]
                
    return sorted(list(found_varietals))

# Apply the function to the 'notes' column
df['varietals'] = df['notes'].apply(extract_varietals)

df.to_csv("data_so_far.csv")


In [ ]:
# one hot encode countries
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

df["countries_extracted"] = df["countries_extracted"].apply(
    lambda x: np.nan if isinstance(x, list) and len(x) == 0 else x
)
df = df.dropna(subset=["countries_extracted"])
one_hot_df = pd.DataFrame(
    mlb.fit_transform(df["countries_extracted"]),
    columns=mlb.classes_,
    index=df.index
)

len(one_hot_df.columns)

In [ ]:
df_num_cols = ["rating", "aroma", "acidity", "body", "flavor", "aftertaste", "agtron_1", "agtron_2", "price_per_oz"]
df[df_num_cols] = df[df_num_cols].replace('', np.nan)

# Now safely convert to float
df_num = df[df_num_cols].astype(float).dropna()

cov_matrix = df_num.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(cov_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.show();

In [ ]:
# let's see if the statistical ratings directly predict the rating

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = np.array(df_num.drop(columns=["rating", "agtron_1", "agtron_2", "price_per_oz"]))
y = np.array(df_num["rating"])

cols = ["aroma", "acidity", "body", "flavor", "aftertaste"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

y_pred_train = lr_model.predict(X_train)
y_pred_test = lr_model.predict(X_test)

def mse(y, y_hat):
    n = y.shape[0]
    return np.sum((y - y_hat) ** 2) / n

print("MSE Train: ", mse(y_train, y_pred_train))
print("MSE Test: ", mse(y_test, y_pred_test))

print(dict(zip(cols, lr_model.coef_)))
print(lr_model.intercept_)


In [ ]:
X = np.array(df_num[["agtron_1", "agtron_2"]])
y = np.array(df_num["rating"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

y_pred_train = lr_model.predict(X_train)
y_pred_test = lr_model.predict(X_test)

print("MSE Train: ", mse(y_train, y_pred_train))
print("MSE Test: ", mse(y_test, y_pred_test))

print(dict(zip(["agrton_1", "agtron_2"], lr_model.coef_)))
print(lr_model.intercept_)

print(f"{y_test[:5]}, {y_pred_test[:5]}")

In [ ]:
import pandas as pd

df = pd.read_csv("reviews_to_2020.csv")
display(df[df["Company"] == "Folgers"])
display(df[df["Company"] == "Folgers"]["URL"])